# SPFI ×4 — training on EMPS (Colab GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SS1162/SR_amat_3/blob/main/notebooks/train_colab.ipynb)

Trains the SPFI model with the repo's `algo/` code **unchanged** (`algo.train.SPFITrainer`) on the
**train split only** of EMPS (`train.csv`). Test images are never extracted.

The result is a checkpoint file `spfi_x4_best.pt` in the same format the local app expects
(`model_state` + `hparams`), so you can use it locally for inference or for fine-tuning.

**Width comparison (C8 vs C16):** set `EMBED_DIM`, `MAX_ITERS` and `PATIENCE` as described under
*Settings*, train once per width, then compare on the validation split with `02_width_eval_val_colab.ipynb`.

**Training recipe:** `RECIPE = "v1"` is the original recipe (one patch per update, constant learning rate)
used for the first C8 / C16 runs; `RECIPE = "v2"` is the improved one (16 patches per update, warm-up +
cosine learning-rate decay, averaged weights, flips / rotations). See *Settings*.

**Before you start:** Runtime → Change runtime type → **GPU**. Then run the cells top to bottom.

In [ ]:
import torch

print("torch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > GPU, then rerun.")


## Settings
For a **fixed-budget width comparison** keep everything identical between runs except `EMBED_DIM`
(the defaults below are the C8 run of the `v2` comparison; for C16 change only `EMBED_DIM`):

| setting | C8 run | C16 run |
|---|---|---|
| `EMBED_DIM` | `8` | `16` |
| `RECIPE` | `"v2"` | `"v2"` |
| `MAX_ITERS` | `10000` (`v1` runs: `5000` / `20000`) | same |
| `MAX_HOURS` | `None` | `None` |
| `PATIENCE` | `1000` (early stopping off) | `1000` |
| `SEED` / `SPLIT_SEED` | `0` / `0` | `0` / `0` |

Both runs then see the same training patches in the same order and the same validation images.

**Training recipe** (`RECIPE`; the same for both widths in a comparison):

| | `"v1"` (original) | `"v2"` (improved) | why |
|---|---|---|---|
| `BATCH_SIZE` | 1 | 16 | each update averages 16 patches: a steadier direction, and 16× more data per update |
| `LR` | 1e-4 | 2e-4 | a bigger batch takes larger steps safely |
| `LR_SCHEDULE` | constant | cosine | warm-up for `WARMUP_ITERS`, then the rate falls smoothly to 1% at `MAX_ITERS`, so the model settles |
| `WARMUP_ITERS` | 0 | 500 | avoids large first steps from random weights |
| `EMA_DECAY` | None | 0.999 | validates / exports an average of the recent weights, which is smoother than the last step |
| `AUGMENT` | False | True | random flips / 90° rotations: 8× more distinct patches from the same 292 images |

`v2` runs get `_v2` in their file names (e.g. `spfi_x4_final_e8_s0_v2_it10000.pt`), so they never overwrite
the `v1` runs. In `02_width_eval_val_colab.ipynb` set `RUN_SUFFIX = "_v2_it10000"` to compare them.
With `v2` one iteration does 16× the work, so 10000 iterations = 160000 patches (8× the 20000-iteration
`v1` runs).

**Long runs and disconnects:** with `SAVE_TO_DRIVE = True` and `RESUME = True`, the full training state
(weights, optimizer, iteration, best checkpoint, random-number state, elapsed time) is saved to Drive every
`RESUME_EVERY` iterations. If Colab disconnects, open the notebook again with the **same settings** and
Run all: training continues from the last saved iteration, exactly as if it had not stopped.

In [ ]:
from pathlib import Path

# ---- Code (the algo/ package is used exactly as it is in the repo) ----
CODE_REPO_URL = "https://github.com/SS1162/SR_amat_3.git"
CODE_BRANCH = "riki/better-training"   # the v2 recipe lives here (main does not have it yet)

# ---- Data: public EMPS release (same images and train/test split as NanoVision's datasets/emps/EMPS) ----
EMPS_COMMIT = "dd8f86d714894fd08f3db80041893982ce058dd6"
EMPS_URL = f"https://codeload.github.com/by256/emps/zip/{EMPS_COMMIT}"
EMPS_SHA256 = "141462d290f9ac780bf9599ef5cb4b3e6708adfe6abafd1d4b9d6c786a6ad3f8"

WORK_DIR = Path("/content")
CODE_DIR = WORK_DIR / "SR_amat_3"
DATA_DIR = WORK_DIR / "emps"
OUTPUT_DIR = WORK_DIR / "output"

# ---- Training recipe: "v1" = original (batch 1, constant LR), "v2" = improved (see the table above) ----
RECIPE = "v2"
if RECIPE == "v1":
    BATCH_SIZE, LR, LR_SCHEDULE, WARMUP_ITERS, EMA_DECAY, AUGMENT = 1, 1e-4, "constant", 0, None, False
elif RECIPE == "v2":
    BATCH_SIZE, LR, LR_SCHEDULE, WARMUP_ITERS, EMA_DECAY, AUGMENT = 16, 2e-4, "cosine", 500, 0.999, True
else:
    raise ValueError(f"unknown RECIPE {RECIPE!r}")
WEIGHT_DECAY = 1e-2
PATIENCE = 1000        # early stopping: validation checks (every 250 iterations) without improvement; 1000 = off
SEED = 0               # model init + training patch order
SPLIT_SEED = 0         # train/val image split; keep it fixed so every run has the same val images
EMBED_DIM = 8          # feature channels (model width): 8 = C8, 16 = C16 (48 = the default in algo/train.py)

# Optional limits so the run ends before Colab disconnects. None = no limit (train until early stopping).
MAX_HOURS = None
MAX_ITERS = 10000      # stop after exactly this many iterations (use it for fixed-budget comparisons)
assert LR_SCHEDULE != "cosine" or MAX_ITERS is not None, "the cosine schedule needs MAX_ITERS (it decays to it)"

# Goes into the output file names, e.g. spfi_x4_final_e8_s0_v2_it10000.pt. v1 runs with MAX_ITERS = 5000 keep
# the shorter name of the first comparison (spfi_x4_final_e8_s0.pt), so those files stay where they are.
RUN_TAG = (f"e{EMBED_DIM}_s{SEED}" + ("" if RECIPE == "v1" else f"_{RECIPE}")
           + (f"_it{MAX_ITERS}" if MAX_ITERS not in (None, 5000) or RECIPE != "v1" else ""))

# Copy the best checkpoint to Google Drive every time it improves, so a disconnect does not lose it.
SAVE_TO_DRIVE = True
DRIVE_DIR = Path("/content/drive/MyDrive/spfi_checkpoints")

# Resume after a disconnect (needs SAVE_TO_DRIVE): the training state is saved to Drive every RESUME_EVERY
# iterations, and a new run with the same settings continues from it.
RESUME = True
RESUME_EVERY = 500


## Code and dependencies

In [ ]:
# torch / torchvision / numpy / Pillow are preinstalled on Colab; add what algo/ needs on top.
!pip install -q "pytorch_wavelets>=1.3" "PyWavelets>=1.6" "setuptools<81"


In [ ]:
import hashlib
import subprocess
import time
import urllib.request
import zipfile


def git(*args, cwd=None, attempts=3):
    """Run git, retrying network hiccups."""
    for attempt in range(1, attempts + 1):
        result = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)
        if result.returncode == 0:
            return result.stdout
        if attempt < attempts:
            time.sleep(5 * attempt)
    raise RuntimeError(f"git {args[0]} failed:\n{result.stderr}")


def download_emps():
    """Download the pinned public EMPS archive once (no credentials needed) and check its SHA-256."""
    archive = DATA_DIR / "emps.zip"
    if not archive.exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        print("Downloading EMPS...")
        urllib.request.urlretrieve(EMPS_URL, archive)
    sha = hashlib.sha256(archive.read_bytes()).hexdigest()
    if sha != EMPS_SHA256:
        print(f"WARNING: archive SHA-256 {sha} differs from the recorded {EMPS_SHA256}")
    return archive


def read_split(text):
    """Image ids (file stems) listed in train.csv / test.csv, one per line."""
    return [line.split(",")[0].strip() for line in text.splitlines() if line.strip()]


def extract_emps_split(split):
    """Extract train.csv, test.csv and only the images of `split` ('train' or 'test') into DATA_DIR/<split>/."""
    dest = DATA_DIR / split
    (dest / "images").mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(download_emps()) as zf:
        root = next(n for n in zf.namelist() if n.endswith("/train.csv"))[:-len("train.csv")]
        for name in ("train.csv", "test.csv"):
            (dest / name).write_bytes(zf.read(root + name))
        ids = read_split((dest / f"{split}.csv").read_text())
        for image_id in ids:
            (dest / "images" / f"{image_id}.png").write_bytes(zf.read(f"{root}images/{image_id}.png"))
    return dest, ids


import sys

# Always check out CODE_BRANCH, also when the folder exists from an earlier run with another branch.
if not (CODE_DIR / ".git").exists():
    git("clone", "--quiet", "--no-checkout", CODE_REPO_URL, str(CODE_DIR))
git("fetch", "--quiet", "--depth", "1", "origin", CODE_BRANCH, cwd=CODE_DIR)
git("checkout", "--quiet", "--force", "--detach", "FETCH_HEAD", cwd=CODE_DIR)
for name in [m for m in sys.modules if m == "algo" or m.startswith("algo.")]:
    del sys.modules[name]   # re-import algo/ from the branch just checked out
print("code:", CODE_BRANCH, "@", git("log", "-1", "--format=%h %s", cwd=CODE_DIR).strip())

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))   # makes `algo` importable
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Dataset
EMPS is downloaded from its public release ([by256/emps](https://github.com/by256/emps), pinned commit),
so no GitHub token is needed. Its `train.csv` / `test.csv` are identical to the ones in NanoVision's
`datasets/emps/EMPS`.

Only the images listed in `train.csv` are extracted.

In [ ]:
EMPS_DIR, train_ids = extract_emps_split("train")
test_ids = read_split((EMPS_DIR / "test.csv").read_text())

on_disk = sorted(p.stem for p in (EMPS_DIR / "images").glob("*.png"))
assert on_disk == sorted(train_ids), "images/ must hold exactly the train split"
assert not set(on_disk) & set(test_ids), "test images must not be used for training"
print(f"{len(on_disk)} train images in {EMPS_DIR / 'images'} (test split: {len(test_ids)} images, not downloaded)")


In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)


## Build the trainer
`SPFITrainer` splits the train images 80/20 into training / validation (for early stopping), so the
validation set also comes from the train split.

In [ ]:
import shutil
from datetime import datetime

from algo.train import SPFITrainer

stamp = f"{RUN_TAG}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
CKPT_DIR = WORK_DIR / "checkpoints" / stamp
RUN_DIR = WORK_DIR / "runs" / stamp       # metrics.jsonl / meta.json / samples/ (same format as the local UI)

trainer = SPFITrainer(
    emps_dir=EMPS_DIR,
    ckpt_dir=CKPT_DIR,
    lr=LR,
    wd=WEIGHT_DECAY,
    patience=PATIENCE,
    seed=SEED,
    split_seed=SPLIT_SEED,
    embed_dim=EMBED_DIM,
    run_dir=RUN_DIR,
    batch_size=BATCH_SIZE,
    lr_schedule=LR_SCHEDULE,
    warmup_iters=WARMUP_ITERS,
    total_iters=MAX_ITERS,
    ema_decay=EMA_DECAY,
    augment=AUGMENT,
    cache_images=True,     # same patches, without re-reading each PNG for every patch
)
print(f"embed_dim {EMBED_DIM}: {sum(p.numel() for p in trainer.model.parameters()):,} parameters")

# ---- resume state (one file per run setting, on Drive) ----
RUN_SETTINGS = {"embed_dim": EMBED_DIM, "seed": SEED, "split_seed": SPLIT_SEED, "lr": LR, "wd": WEIGHT_DECAY,
                "patience": PATIENCE, "max_iters": MAX_ITERS}
if RECIPE != "v1":   # v1 keeps the old settings dict, so its earlier resume files still match
    RUN_SETTINGS.update(recipe=RECIPE, batch_size=BATCH_SIZE, lr_schedule=LR_SCHEDULE, warmup_iters=WARMUP_ITERS,
                        ema_decay=EMA_DECAY, augment=AUGMENT)
RESUME_FILE = DRIVE_DIR / f"resume_{RUN_TAG}_it{MAX_ITERS}.pt"
RESUMED_FROM, PRIOR_SECONDS = 0, 0.0


def save_resume(elapsed_seconds):
    """Everything needed to continue exactly where training is now (called between iterations)."""
    best = CKPT_DIR / "spfi_x4_best.pt"
    state = {
        "settings": RUN_SETTINGS,
        "iteration": trainer.iteration,
        "model_state": trainer.model.state_dict(),
        "optimizer_state": trainer.optimizer.state_dict(),
        "ema_state": trainer.ema_model.state_dict() if trainer.ema_model is not None else None,
        "best_val_loss": trainer.best_val_loss,
        "no_improve": trainer.no_improve,
        "best_checkpoint": torch.load(best, map_location="cpu", weights_only=True) if best.exists() else None,
        "train_rng": trainer._train_rng.getstate(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "elapsed_seconds": elapsed_seconds,
    }
    tmp = WORK_DIR / f"{RESUME_FILE.name}.tmp"
    torch.save(state, tmp)
    shutil.copy2(tmp, RESUME_FILE)   # write locally first, then copy: a disconnect mid-copy leaves the old file
    # Training curves of this session (train loss every 10 iterations, validation every 250), so a disconnect
    # does not lose them; 02_width_eval_val_colab.ipynb joins the files of all sessions of a run.
    for name in ("metrics.jsonl", "meta.json"):
        if (RUN_DIR / name).exists():
            shutil.copy2(RUN_DIR / name, DRIVE_DIR / f"{Path(name).stem}_{stamp}{Path(name).suffix}")


if RESUME and SAVE_TO_DRIVE and RESUME_FILE.exists():
    state = torch.load(RESUME_FILE, map_location=trainer.device, weights_only=False)
    assert state["settings"] == RUN_SETTINGS, f"{RESUME_FILE.name} was saved with other settings: {state['settings']}"
    trainer.model.load_state_dict(state["model_state"])
    trainer.optimizer.load_state_dict(state["optimizer_state"])
    if state.get("ema_state") is not None:
        trainer.ema_model.load_state_dict(state["ema_state"])
    trainer.iteration = state["iteration"]
    trainer.best_val_loss = state["best_val_loss"]
    trainer.no_improve = state["no_improve"]
    if state["best_checkpoint"] is not None:
        torch.save(state["best_checkpoint"], CKPT_DIR / "spfi_x4_best.pt")
    trainer._train_rng.setstate(state["train_rng"])
    torch.set_rng_state(state["torch_rng"].cpu())
    if state["cuda_rng"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in state["cuda_rng"]])
    RESUMED_FROM, PRIOR_SECONDS = state["iteration"], state["elapsed_seconds"]
    print(f"RESUMED from {RESUME_FILE.name}: iteration {RESUMED_FROM}, "
          f"{PRIOR_SECONDS / 60:.1f} min already trained, best val loss {trainer.best_val_loss:.4f}")
elif RESUME and SAVE_TO_DRIVE:
    print(f"no resume file ({RESUME_FILE.name}): starting from iteration 0")


## Speed and memory check
A few training-sized forward + backward passes (random inputs; the weights, optimizer and patch order are not
touched) to show the GPU memory and time per iteration with this `BATCH_SIZE`, and the expected run time.
If it runs out of memory, lower `BATCH_SIZE` (e.g. 8) and keep the same value for both widths.

In [ ]:
from algo.train import HR_PATCH, LR_PATCH, total_loss

rng_state = torch.get_rng_state()
cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
x = torch.rand(BATCH_SIZE, 1, LR_PATCH, LR_PATCH, device=trainer.device)
y = torch.rand(BATCH_SIZE, 1, HR_PATCH, HR_PATCH, device=trainer.device)
times = []
for _ in range(4):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    total_loss(trainer.model(x), y).backward()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    times.append(time.time() - t0)
trainer.model.zero_grad(set_to_none=True)
torch.set_rng_state(rng_state)
if cuda_state is not None:
    torch.cuda.set_rng_state_all(cuda_state)
step = min(times[1:])   # the first pass includes start-up
print(f"batch {BATCH_SIZE}: {step:.3f} s per forward+backward (model only, no data loading)")
if torch.cuda.is_available():
    peak, total = torch.cuda.max_memory_allocated() / 2**30, torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f"peak GPU memory {peak:.2f} GB of {total:.1f} GB")
if MAX_ITERS is not None:
    left = MAX_ITERS - trainer.iteration
    print(f"{left:,} iterations left: at least ~{left * step / 3600:.1f} h of model time (data loading and "
          "validation add to it; the [iter] lines show the real pace)")

## Train
Runs until early stopping, `MAX_HOURS` or `MAX_ITERS`, whichever comes first. A checkpoint is saved and
the model is validated every 250 iterations.

You can also press **Stop** (■) at any time: the trainer stops cleanly and you can go on to *Export*.
Running this cell again continues from where it stopped.

In [ ]:
import threading
import time

stop_event = threading.Event()
done = threading.Event()
best_ckpt = CKPT_DIR / "spfi_x4_best.pt"


def watch(started=time.time()):
    """Stop the trainer at the time limit and mirror the best checkpoint to Drive."""
    mirrored = None
    while not done.wait(5):
        if MAX_HOURS is not None and time.time() - started >= MAX_HOURS * 3600:
            stop_event.set()
        if SAVE_TO_DRIVE and best_ckpt.exists():
            mtime = best_ckpt.stat().st_mtime
            if mtime != mirrored:
                time.sleep(1)   # let torch.save finish writing
                shutil.copy2(best_ckpt, DRIVE_DIR / f"spfi_x4_best_{stamp}.pt")
                mirrored = mtime


watcher = threading.Thread(target=watch, daemon=True)
watcher.start()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
train_started = time.time()
saving = RESUME and SAVE_TO_DRIVE
try:
    # Train in chunks of RESUME_EVERY iterations (exactly the same updates as one long call); between
    # chunks the full state goes to Drive so a disconnect loses at most one chunk.
    while not stop_event.is_set() and (MAX_ITERS is None or trainer.iteration < MAX_ITERS):
        target = trainer.iteration + (RESUME_EVERY if saving else 10**12)
        if MAX_ITERS is not None:
            target = min(target, MAX_ITERS)
        trainer.train(stop_event=stop_event, max_iters=target)
        if trainer.iteration < target:   # early stopping, stop button or time limit
            break
        if saving:
            save_resume(PRIOR_SECONDS + time.time() - train_started)
            print(f"[resume] state saved at iteration {trainer.iteration}")
finally:
    train_seconds = PRIOR_SECONDS + time.time() - train_started
    PRIOR_SECONDS = train_seconds   # running this cell again (after Stop) keeps counting from here
    done.set()
    watcher.join()
print(f"finished at iteration {trainer.iteration}, best val loss {trainer.best_val_loss:.4f}, "
      f"{train_seconds / 60:.1f} min" + (f" (resumed from iteration {RESUMED_FROM})" if RESUMED_FROM else ""))
REACHED_MAX_ITERS = MAX_ITERS is None or trainer.iteration >= MAX_ITERS
if not REACHED_MAX_ITERS:
    print(f"\n*** WARNING: stopped at iteration {trainer.iteration} of MAX_ITERS={MAX_ITERS} "
          "(stop button, interrupt or time limit). ***\n"
          "*** This run is NOT usable for a fixed-budget comparison"
          + (": run the notebook again with the same settings to resume. ***" if saving else ": run it again. ***"))

## Export the weights
Saves the current weights as `spfi_x4_final_<tag>.pt` and the lowest-validation-loss weights as
`spfi_x4_best_<tag>.pt` (`<tag>` = `e<EMBED_DIM>_s<SEED>`), checks that they load the way the local app loads
them, and writes `train_info_<tag>.json` (code commit, settings, iterations, training time, peak GPU memory).
For a fixed-budget comparison, compare the **final** files: both were trained for exactly `MAX_ITERS`.

In [ ]:
import json

trainer._save_checkpoint("final")   # same format as the periodic checkpoints
final_ckpt = CKPT_DIR / "spfi_x4_final.pt"

# Same checks as the local app's inference loader (inference/runner.py::_load_model)
from algo.model import build_spfi

EXPORTS = {}
for kind, src_ckpt in (("best", best_ckpt), ("final", final_ckpt)):
    if not src_ckpt.exists():
        continue
    dst = OUTPUT_DIR / f"spfi_x4_{kind}_{RUN_TAG}.pt"
    shutil.copy2(src_ckpt, dst)
    ckpt = torch.load(dst, map_location="cpu", weights_only=True)
    hp = ckpt["hparams"]
    build_spfi(scale=hp["scale"], embed_dim=hp["embed_dim"]).load_state_dict(ckpt["model_state"])
    EXPORTS[kind] = dst
    print(f"{dst.name}: iteration {ckpt['iteration']}, best val loss so far {ckpt['best_val_loss']:.4f}, hparams {hp}")
WEIGHTS_FILE = EXPORTS.get("best", EXPORTS["final"])

train_info = {
    "run_tag": RUN_TAG,
    "code_branch": CODE_BRANCH,
    "code_commit": git("rev-parse", "HEAD", cwd=CODE_DIR).strip(),
    "embed_dim": EMBED_DIM,
    "params": sum(p.numel() for p in trainer.model.parameters()),
    "seed": SEED,
    "split_seed": SPLIT_SEED,
    "recipe": RECIPE,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "lr_schedule": LR_SCHEDULE,
    "warmup_iters": WARMUP_ITERS,
    "ema_decay": EMA_DECAY,
    "augment": AUGMENT,
    "patches_seen": trainer.iteration * BATCH_SIZE,
    "weight_decay": WEIGHT_DECAY,
    "patience": PATIENCE,
    "max_iters": MAX_ITERS,
    "max_hours": MAX_HOURS,
    "iterations": trainer.iteration,
    "reached_max_iters": REACHED_MAX_ITERS,
    "resumed_from_iteration": RESUMED_FROM,
    "best_val_loss": trainer.best_val_loss if trainer.best_val_loss != float("inf") else None,   # None: no validation yet
    "train_seconds": round(train_seconds, 1),
    "seconds_per_iter": round(train_seconds / max(trainer.iteration, 1), 4),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "peak_gpu_memory_mb": round(torch.cuda.max_memory_allocated() / 2**20, 1) if torch.cuda.is_available() else None,   # this session
    "torch": torch.__version__,
}
TRAIN_INFO_FILE = OUTPUT_DIR / f"train_info_{RUN_TAG}.json"
TRAIN_INFO_FILE.write_text(json.dumps(train_info, indent=2))
print(json.dumps(train_info, indent=2))

logs_zip = shutil.make_archive(str(OUTPUT_DIR / f"spfi_run_{stamp}"), "zip", RUN_DIR)
if SAVE_TO_DRIVE:
    for f in (*EXPORTS.values(), TRAIN_INFO_FILE, Path(logs_zip)):
        shutil.copy2(f, DRIVE_DIR)
    print("copied to", DRIVE_DIR)

In [ ]:
from google.colab import files

for f in EXPORTS.values():
    files.download(str(f))          # the weights files (best and final)
files.download(str(TRAIN_INFO_FILE))   # settings + training time / memory, used by the comparison notebook
files.download(logs_zip)            # optional: training curves / samples (unzip into the local runs/ folder)

## Using the weights locally
* **Inference tab** of the local app: set the model path to the downloaded `spfi_x4_best.pt`.
* **Fine-tuning:** `python -m algo.train --emps_dir <EMPS> --init finetune --weights spfi_x4_best.pt`.
* **Width comparison on the validation split:** `02_width_eval_val_colab.ipynb` (loads `spfi_x4_final_e8_s0.pt` and `spfi_x4_final_e16_s0.pt`).
* **Test PSNR / SSIM / FLOPs:** `inference_colab.ipynb` (final evaluation only; the test split stays locked).